# Day 1 — Setup on Google Colab

Runs the Day 1 checks for the **inverse scattering** dataset:
1. GPU check  2. Mount Google Drive  3. Get the code  4. Get the data  5. Run the checks

**Before running:** `Runtime → Change runtime type → T4 GPU`, then edit the config cell below and run all cells in order (`Runtime → Run all`).

- Code comes from GitHub (pulled fresh every run).
- Data and results are stored in Google Drive at `MyDrive/thesis/`, so they survive Colab restarts.

In [ ]:
NOTEBOOK_VERSION = "2026-10-16b"   # if you don't see this printed, you are running an old copy
# ===== CONFIG: edit this =====
GITHUB_REPO = "YOUR_USERNAME/thesis-gs"   # your GitHub repo (owner/name)
BRANCH      = "main"
DRIVE_ROOT  = "/content/drive/MyDrive/thesis"   # data + outputs are kept here
# =============================
print("Notebook version:", NOTEBOOK_VERSION)


## 1. GPU check

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU. Go to Runtime -> Change runtime type -> T4 GPU, then run again.")

## 2. Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/outputs", exist_ok=True)
print("Drive folder ready:", DRIVE_ROOT)

## 3. Get the code
Clones (or updates) **our repo** and the **InverseBench** repo.
If our repo is **private**, add a GitHub token in Colab: click the 🔑 *Secrets* icon on the left → add `GITHUB_TOKEN` → enable notebook access.

In [ ]:
import os, subprocess, sys

token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass  # no secret set -> fine for public repos

CODE = "/content/thesis-gs"
url = f"https://{token}@github.com/{GITHUB_REPO}.git" if token else f"https://github.com/{GITHUB_REPO}.git"

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = r.stdout + r.stderr
        if token:
            msg = msg.replace(token, "***")
        print("ERROR running:", " ".join(cmd[:4]), "...\n", msg[-2000:])
        raise SystemExit("Copy the error above and send it to Claude.")
    return r.stdout

# 1. our repo (fresh copy every session)
if os.path.isdir(f"{CODE}/.git"):
    run(["git", "-C", CODE, "pull", "--ff-only"])
else:
    run(["git", "clone", "-b", BRANCH, url, CODE])

# 2. InverseBench (for the physics forward model)
IB = "/content/InverseBench"
if not os.path.isdir(IB):
    run(["git", "clone", "--depth", "1", "https://github.com/devzhk/InverseBench.git", IB])

# 3. the only extra package we need (torch/numpy/scipy/matplotlib are preinstalled)
run([sys.executable, "-m", "pip", "install", "-q", "lmdb"])
import lmdb

# 4. sanity checks
print("Repo files   :", sorted(f for f in os.listdir(CODE) if not f.startswith(".")))
assert os.path.isfile(f"{CODE}/day1/load_scatter_data.py"), (
    "day1/load_scatter_data.py not found -> the repo layout is wrong (maybe nested one folder too deep).")
print("Latest commit:", run(["git", "-C", CODE, "log", "-1", "--oneline"]).strip())
print("lmdb         :", lmdb.version())
print("Code ready.")


## 4. Get the inverse scattering data
Downloads the zips from CaltechDATA **once** into Drive, then unzips them to the fast local disk each session.

In [ ]:
import glob, shutil, urllib.request, zipfile

BASE = "https://data.caltech.edu/records/zg89b-mpv16/files"
LOCAL_DATA = "/content/data"
os.makedirs(LOCAL_DATA, exist_ok=True)

for name in ["inv-scatter-test", "inv-scatter-val"]:
    zip_path = f"{DRIVE_ROOT}/data/{name}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {name}.zip ...")
        try:
            urllib.request.urlretrieve(f"{BASE}/{name}.zip?download=1", zip_path)
        except Exception as e:
            print(f"  Download failed ({e}).")
            print(f"  Manual fix: download {name}.zip from https://data.caltech.edu/records/zg89b-mpv16")
            print(f"  and upload it to Google Drive at {zip_path}, then rerun this cell.")
            if os.path.exists(zip_path):
                os.remove(zip_path)  # remove partial download
            raise SystemExit("Data download failed - see message above.")
    target = f"{LOCAL_DATA}/{name}"
    if not os.path.exists(f"{target}/data.mdb"):
        tmp = f"/content/_unzip_{name}"
        shutil.rmtree(tmp, ignore_errors=True)
        zipfile.ZipFile(zip_path).extractall(tmp)
        mdb = glob.glob(f"{tmp}/**/data.mdb", recursive=True)
        assert mdb, f"No data.mdb found inside {name}.zip"
        shutil.rmtree(target, ignore_errors=True)
        shutil.move(os.path.dirname(mdb[0]), target)
        shutil.rmtree(tmp, ignore_errors=True)
    print(f"{name}: ready at {target}  ->  {os.listdir(target)}")
assert os.path.exists(f"{LOCAL_DATA}/inv-scatter-test/data.mdb"), "inv-scatter-test data is missing"
print("Data ready.")


## 5. Run the Day 1 checks

In [ ]:
import subprocess, sys

def run_script(args):
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise SystemExit("The script above failed - copy its output and send it to Claude.")

run_script([f"{CODE}/day1/check_env.py"])


In [ ]:
OUT = f"{DRIVE_ROOT}/outputs/day1"
run_script([f"{CODE}/day1/load_scatter_data.py",
            "--data", f"{LOCAL_DATA}/inv-scatter-test",
            "--repo", IB,
            "--out", OUT])


In [ ]:
from IPython.display import Image, display
for f in ["cells_grid.png", "measurements.png"]:
    path = f"{OUT}/{f}"
    if os.path.exists(path):
        print(f)
        display(Image(filename=path))
    else:
        print(f"{f} was not created - the loader cell above failed; check its output.")


## Done ✅
Expected: 100 entries, values 0..1, measurements of shape `(1, 20, 360)`, consistency ~1e-10, and the two images above.
The images are also saved in Google Drive under `MyDrive/thesis/outputs/day1/`.

**To share results with Claude:** copy the printed output of the two scripts and send the two images.